# Dashboard Interativo dos Campeões — Dissecação Final dos Vencedores PAC (Pós-Curadoria)

**Este notebook não é mais um passo prévio ao pipeline** (a antiga exploração visual "às cegas", anterior à triagem automática, foi aposentada). Ele é a camada final: um dashboard que consome diretamente `candidatos_vencedores_OURO_PURIFICADO.csv` (com fallback para `candidatos_vencedores_OURO.csv`, depois `candidatos_vencedores_consolidados.csv`, se os anteriores não existirem) — os eventos de acoplamento teta→gama/HG que já passaram pelos portões:

- **Estatístico**: `veredito_refino == 'Candidato robusto'` (FDR + ajuste Gama + sem suspeita de banda larga).
- **FOOOF**: erro de ajuste < 0.15, pico periódico real (`cf_*_fooof_v2` não-nulo) nas duas bandas. `knee_valido_teta/gamma` NÃO é exigido (é diagnóstico de identificabilidade do modelo, não de autenticidade de PAC — auditoria 2026-09, ver `consolida_vencedores.py`).
- **Harmônico**: `veredito_harmonico == 'CLEAN'` OU `'REVISAR_RAZAO_INTEIRA (Nx)'` (coincidência numérica de frequência sem confirmação de travamento de fase/PLV). Só reprova por evidência real de contaminação (PLV>0.8, ambiguidade de múltiplos N, ou assimetria de teta) — auditoria 2026-09.
- **Comportamental**: janela anotada, excluindo `Artefato / Cabo`.
- **Robustez + transiente** (`robustez_parametros.py`): sobrevive por maioria (≥4/6 valores de n_bins, ≥1/2 mapas recomputados estáveis) ao sweep de parâmetros, sem MVL/Rayleigh circular, sem transiente mecânico, sem correlação suspeita de banda larga. **Pegada espacial (`footprint_*`) é só informativa, não filtra** — o projeto não tem mapa de geometria/adjacência dos 32 canais para decidir com segurança o que é acoplamento amplamente coerente (real) vs. artefato difuso (ver limitação documentada em `audita_footprint.py`).
- **Notch multi-harmônico** (só no `OURO_PURIFICADO.csv`): auditoria 2026-09 descobriu que `triagem_pac.py`/`refina_candidatos.py` nunca aplicavam notch de 60/120/180/240Hz apesar da regra #1 do `CLAUDE.md` — o ruído de linha bruto chega a ~4600x o fundo em 240Hz. Recalculado `z` com notch (`n_surr=1000`, média de 5 tiragens para os eventos na zona cinza z∈[1.5,4.5]) e mantidos só os que sobrevivem com `z_médio ≥ 3.0`: **227 dos 278 (81.7%)**. Ver `resultados/_verificacao_notch_278.csv` (estatística agregada) e `_notch_verificacao_detalhada.csv` (as 5 tiragens por evento da zona cinza). **Limitação registrada, não resolvida**: esse portão só reavalia os candidatos que já chegaram ao OURO.csv — `triagem_pac.py` histórico rodou sem notch, então é possível que candidatos tenham sido perdidos já na varredura inicial por causa do ruído de linha, antes de chegarem perto desta lista.

**Uso:** escolha **Rato → Comportamento → Janela campeã** (ordenada por Z-score) nos controles abaixo e clique em **Visualizar** para gerar a dissecação em 5 painéis daquele evento específico:

1. **LFP bruto + banda teta (4–8 Hz) + banda gama (30–80 Hz)** — traçado temporal da janela de 10s.
2. **Timeline descritiva** — envelope de teta x MI cru em janela deslizante de 0.75s (sem teste estatístico; só indica onde a modulação parece se concentrar, não prova nada sozinha — ver auditoria 2026-09 sobre falsos picos por amostra pequena).
3. **FOOOF banda baixa (2–45 Hz)** — fundo aperiódico (knee) e pico de teta destacado.
4. **FOOOF banda alta (35 Hz–~0,95×Nyquist)** — com limpeza de ruído de linha Kuhn aplicada antes do ajuste, picos de gama/HG destacados. Eixo Y igualado entre os painéis 3 e 4.
5. **Comodulograma fase×amplitude** da janela de 10s, centralizado e reduzido — branco = célula não-significativa após FDR (não ausência de dado). **Usa `seg_limpo` (com notch) desde a auditoria 2026-09** — antes desse fix, o mapa mostrava manchas fantasma de altíssimo z (até 59) exatamente nas células de amplitude que encostam nos harmônicos de linha (ex. 235-255Hz straddling o notch de 240Hz).

*Nota de reprodutibilidade*: o comodulograma recalculado aqui usa surrogates com seed não fixada por padrão neste notebook exploratório, então o z-score de pico pode divergir um pouco do `z_score_refinado` gravado no CSV (mesma limitação já documentada em `comodulogram_interativo.py`) — inclusive já observamos o mesmo evento variar entre 0 e várias células significativas em execuções sucessivas. Os valores de FOOOF (cf_teta, cf_gama, R², knee), por serem determinísticos, devem bater com as colunas `*_fooof_v2`/`r2_*_fooof` do dataset mestre.


In [ ]:
# CELULA 1: IMPORTS CANONICOS E CONFIGURACAO
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import hilbert
import ipywidgets as widgets
from IPython.display import display, clear_output

# Localiza a raiz SCRIPT subindo os diretorios ate achar pac_core
cur = os.path.abspath(os.getcwd())
while cur and os.path.dirname(cur) != cur:
    if os.path.isdir(os.path.join(cur, 'pac_core')):
        break
    cur = os.path.dirname(cur)
SCRIPT_DIR = cur
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# Nucleo compartilhado
from pac_core.io import fatia_janela, concatena_sessao
from pac_core.filtering import filtra_sinal, aplica_notch
from pac_core.pac_metrics import fase_para_bin_idx, _mi_de_bin_idx

# Nucleo do comodulograma
from pipeline.etapa3_comodulograma.comodulogram import (
    calcula_comodulograma_z, z_pico_par, bh_fdr_mapa, p_valores_por_celula,
    FASES_DEFAULT, AMPS_DEFAULT,
)

# Ajuste FOOOF + painel
from pipeline.etapa5_exploracao.comodulogram_interativo import (
    ajusta_fooof_teta_gamma, painel_fooof,
    FOOOF_TETA_FIT_RANGE, FOOOF_CTX_S,
    N_SURR, N_BINS, FDR_Q,
)

BASE_LAC_NOCI = os.path.join(os.path.dirname(SCRIPT_DIR), 'LAC_NOCI')
# Prioridade: OURO_PURIFICADO_v2.csv (190 -- refino E robustez com notch
# multi-harmonico, auditoria 2026-09) > OURO_PURIFICADO.csv (227 -- so
# refino com notch, robustez_parametros.py ainda so filtrava 60Hz) >
# OURO.csv (278, sem nenhum portao de notch) > consolidados.csv.
candidatos_csv = [
    os.path.join(SCRIPT_DIR, 'resultados', 'candidatos_vencedores_OURO_PURIFICADO_v2.csv'),
    os.path.join(SCRIPT_DIR, 'resultados', 'candidatos_vencedores_OURO_PURIFICADO.csv'),
    os.path.join(SCRIPT_DIR, 'resultados', 'candidatos_vencedores_OURO.csv'),
    os.path.join(SCRIPT_DIR, 'resultados', 'candidatos_vencedores_consolidados.csv'),
    os.path.join(SCRIPT_DIR, 'candidatos_vencedores_consolidados.csv'),
]
CSV_VENCEDORES = next((c for c in candidatos_csv if os.path.isfile(c)), candidatos_csv[0])

print('SCRIPT_DIR      :', SCRIPT_DIR)
print('BASE_LAC_NOCI   :', BASE_LAC_NOCI, '(existe:', os.path.isdir(BASE_LAC_NOCI), ')')
print('CSV_VENCEDORES  :', CSV_VENCEDORES, '(existe:', os.path.isfile(CSV_VENCEDORES), ')')

_nome_atual = os.path.basename(CSV_VENCEDORES)
_gates_completos = {'candidatos_vencedores_OURO_PURIFICADO_v2.csv'}
_gates_parciais = {'candidatos_vencedores_OURO_PURIFICADO.csv', 'candidatos_vencedores_OURO.csv'}
if _nome_atual in _gates_parciais:
    print('[AVISO] OURO_PURIFICADO_v2.csv nao encontrado -- usando uma lista '
          'SEM o portao de notch multi-harmonico completo em robustez_parametros.py '
          '(auditoria 2026-09: esse script so filtrava 60Hz, nao 120/180/240Hz -- '
          '16.3% dos 227 mudavam de veredito com o notch completo, a maioria '
          'perdendo robustez por deteccao de transiente antes mascarada). Rode a '
          'purificacao v2 pra lista definitiva de 190.')
elif _nome_atual not in _gates_completos:
    print('[AVISO] Nenhum OURO_PURIFICADO encontrado -- usando lista SEM os '
          'portoes de robustez/transiente/notch (pode incluir falso-positivo '
          'tipo transiente mecanico ou ruido de linha disfarcado de PAC).')


In [22]:
# CELULA 2: CARREGA OS VENCEDORES CONSOLIDADOS E COLAPSA PSEUDOREPLICACAO ESPACIAL
df_vencedores = pd.read_csv(CSV_VENCEDORES)

# Convencao do projeto (ver CLAUDE.md #3): canais vizinhos com o mesmo pico
# na mesma janela sao UM UNICO evento biologico, nao descobertas independentes.
# Colapsa por (sessao, arquivo, janela_ini_s, janela_fim_s), mantendo a
# linha de maior z_score_refinado como representante do evento.
CHAVE_JANELA = ["sessao", "arquivo", "janela_ini_s", "janela_fim_s"]
df_eventos = (
    df_vencedores.sort_values("z_score_refinado", ascending=False)
    .drop_duplicates(CHAVE_JANELA)
    .reset_index(drop=True)
)

print(f"Linhas brutas (canal x par): {len(df_vencedores)}")
print(f"Eventos unicos (pos-colapso espacial): {len(df_eventos)}")
print()
print("Ratos:", sorted(df_eventos['rato'].dropna().unique()))
print("Comportamentos:", sorted(df_eventos['comportamento'].dropna().unique()))

df_eventos.head()


Linhas brutas (canal x par): 278
Eventos unicos (pos-colapso espacial): 278

Ratos: ['MTESC03', 'MTESC04', 'MTESC05']
Comportamentos: ['Exploração / Locomoção', 'Grooming / Limpeza', 'Imóvel / Descanso', 'Movimento de Cabeça', 'Rearing / Em pé', 'Sniffing / Farejando']


,par,arquivo,canal,janela_ini_s,janela_fim_s,z_triagem,mi_observado,z_score_refinado,p_analitico,n_canais_simultaneos,...,robustez_z_min_larguras,robustez_z_mvl,robustez_transiente,robustez_max_diff_transiente_sigma,robustez_suspeito_banda_larga,footprint_n_canais,footprint_n_z_ge3,footprint_frac_z_ge3,footprint_z_canal_alvo,footprint_rank_canal_alvo
0,theta_hg,20240708-123605-001.ns2,30,160.0,170.0,11.6177,0.004634,11.5032,8.442370e-10,1,...,0.01,3.73,False,5.54,False,32,2,0.062,7.08,2
1,theta_hg,20240709-141215-001.ns2,14,45.0,55.0,10.3517,0.002724,10.4679,6.920901e-09,1,...,1.72,4.39,False,6.21,False,16,3,0.188,6.87,1
2,theta_hg,20240711-121046-003.ns2,16,205.0,215.0,11.2786,0.003601,9.9195,1.064090e-08,1,...,3.92,2.16,False,6.57,False,16,4,0.250,8.53,1
3,theta_hg,20240711-121046-002.ns2,14,270.0,280.0,9.5160,0.003179,9.7356,9.689875e-09,1,...,0.20,4.55,False,5.88,False,16,3,0.188,7.57,2
4,theta_gamma,20240711-121046-001.ns2,13,45.0,55.0,9.2739,0.003316,9.3587,2.043546e-07,1,...,0.65,2.09,False,5.66,False,16,2,0.125,4.50,1


In [ ]:
# CELULA 3: RESOLUCAO DE CAMINHO + FIGURA DE DISSECACAO EM 5 PAINEIS

def resolve_pasta_basal(sessao_str, arquivo=None):
    """Localiza a pasta que contem os .ns2 correspondente a um valor da
    coluna 'sessao' do dataset mestre.

    Busca dentro de LAC_NOCI/*/ (sem assumir de antemao se e grupo NOCI ou
    LAC) para nao depender de um mapa rato->pasta hardcoded, que quebraria
    silenciosamente se um novo rato/grupo for adicionado.

    ATENCAO (2 armadilhas reais do dado bruto, nao do codigo):
    1) nomes de sessao como "Rodada-1-02-05-2024" NAO sao unicos -- existem
       pastas com o MESMO nome em MTESC03_LAC e MTESC05_LAC (ratos
       diferentes rodados no mesmo protocolo/data). Se `arquivo` for
       informado e houver mais de um candidato, desempata escolhendo a
       pasta que realmente contem esse arquivo .ns2.
    2) a estrutura de pastas do grupo LAC e inconsistente entre datas: a
       maioria tem uma subpasta "Basal antes da infusao", mas em algumas
       (ex.: MTESC05_LAC/Rodada-1-04-05-2024, Rodada-2-06-05-2024,
       Rodada-2-09-05-2024) os .ns2 ficam direto na pasta da sessao --
       por isso tenta as duas formas antes de desistir.
    """
    nome_pasta = sessao_str
    sufixo = "_Basal antes da infusao"
    if nome_pasta.endswith(sufixo):
        nome_pasta = nome_pasta[: -len(sufixo)]

    candidatos = glob.glob(os.path.join(BASE_LAC_NOCI, "*", nome_pasta, "Basal antes da infusao"))
    candidatos += [c for c in glob.glob(os.path.join(BASE_LAC_NOCI, "*", nome_pasta)) if c not in candidatos]
    if not candidatos:
        raise FileNotFoundError(
            f"Pasta nao encontrada para sessao={sessao_str!r} "
            f"(procurado: {nome_pasta!r} dentro de {BASE_LAC_NOCI})"
        )
    if len(candidatos) == 1 or arquivo is None:
        return candidatos[0]

    for c in candidatos:
        if os.path.isfile(os.path.join(c, arquivo)):
            return c
    raise FileNotFoundError(
        f"sessao={sessao_str!r} existe em {len(candidatos)} pastas "
        f"({candidatos}), mas nenhuma contem o arquivo {arquivo!r}"
    )


def mi_timeline(seg_limpo, fs, f_pico, a_pico, meia_fase=1.0, meia_amp=5.0,
               janela_s=0.75, passo_s=0.1, n_bins=N_BINS):
    """MI cru (sem surrogates) em janela deslizante -- so DESCRITIVO, pra
    localizar no tempo onde a modulacao se concentra. Nao substitui o teste
    estatistico (z_score_refinado/robustez_parametros.py); reusa
    fase_para_bin_idx/_mi_de_bin_idx de pac_core.pac_metrics (mesmo nucleo
    usado no resto do pipeline) em vez de recalcular do zero.

    janela_s=0.75 (~5 ciclos de teta a 7Hz): ATENCAO -- e um trade-off
    deliberado de resolucao temporal por ruido (janela mais curta = MI mais
    instavel/ruidoso ponto a ponto), escolhido pra minimizar o efeito de
    suavizacao/atraso aparente entre o envelope (instantaneo) e o MI
    (sempre uma media sobre a janela) quando comparados lado a lado no
    mesmo grafico -- ainda assim nao elimina esse efeito, so reduz.
    """
    lfp_fase = filtra_sinal(seg_limpo, f_pico - meia_fase, f_pico + meia_fase, fs)
    fase = np.angle(hilbert(lfp_fase))
    lfp_amp = filtra_sinal(seg_limpo, a_pico - meia_amp, a_pico + meia_amp, fs)
    env = np.abs(hilbert(lfp_amp))
    bin_idx = fase_para_bin_idx(fase, n_bins)

    win = int(janela_s * fs)
    passo = max(1, int(passo_s * fs))
    tempos, mi_vals = [], []
    for ini in range(0, len(seg_limpo) - win + 1, passo):
        fim = ini + win
        mi_vals.append(_mi_de_bin_idx(bin_idx[ini:fim], env[ini:fim], n_bins))
        tempos.append((ini + fim) / 2.0 / fs)
    return np.array(tempos), np.array(mi_vals)


def plota_dissecacao_completa(row):
    """Gera a figura de 5 paineis (LFP, timeline teta+MI, FOOOF teta,
    FOOOF gama/HG, comodulograma) para uma linha de df_eventos."""
    pasta_basal = resolve_pasta_basal(row["sessao"], row["arquivo"])
    dados, fs, ids_canais, offsets = concatena_sessao(pasta_basal)

    # canal na convencao do dataset mestre (1-based, ver processa_sessao.py
    # -- chan{indice_array+1}); ver correcao do off-by-one em comodulogram_interativo.py
    canal_idx = int(row["canal"]) - 1
    if not (0 <= canal_idx < dados.shape[1]):
        raise ValueError(f"Canal {row['canal']} (indice {canal_idx}) fora do range "
                         f"[0,{dados.shape[1]}) para {pasta_basal}")

    arquivos = sorted(f for f in os.listdir(pasta_basal)
                      if f.lower().endswith((".ns2", ".bin", ".dat")))
    idx_arquivo = arquivos.index(row["arquivo"])
    offset_s = offsets[idx_arquivo] / fs

    janela_ini_local = float(row["janela_ini_s"])
    janela_fim_local = float(row["janela_fim_s"])
    t_center = offset_s + (janela_ini_local + janela_fim_local) / 2.0
    par_ativo = row["par"]
    fp_evento = float(row["fase_pico_hz"])
    fa_evento = float(row["amp_pico_hz"])

    # --- janela de 10s: LFP bruto/teta/gama + comodulograma -----------------
    seg = fatia_janela(dados, fs, t_center - 5, t_center + 5)[:, canal_idx].astype(float)
    t_seg = np.arange(len(seg)) / fs

    # notch antes de filtrar p/ exibicao (regra do projeto: "Notch 60Hz em
    # tudo" -- sem isso o harmonico de 60/120/180/240Hz aparece como falsa
    # oscilacao gama/HG/HFO). CRITICO: seg_limpo tambem alimenta o
    # comodulograma abaixo (bug corrigido 2026-09 -- antes o comodulograma
    # usava `seg` cru, o que produzia manchas "fantasma" de altissimo z
    # exatamente nas celulas de amplitude que encostam nos harmonicos de
    # linha, ex. 235-255Hz straddling o notch de 240Hz; z chegou a 59 num
    # caso, caindo pra 5.8 no mesmo evento so de trocar seg->seg_limpo).
    seg_limpo = aplica_notch(seg, fs, freqs_notch=[60, 120, 180, 240])
    seg_teta = filtra_sinal(seg_limpo, 4, 8, fs)
    seg_gama = filtra_sinal(seg_limpo, 30, 80, fs)

    # --- contexto de 45s p/ FOOOF (mesma janela usada por enriquece_dataset_mestre.py) --
    t_total_s = dados.shape[0] / fs
    ctx_ini = max(0.0, t_center - FOOOF_CTX_S / 2)
    ctx_fim = min(t_total_s, t_center + FOOOF_CTX_S / 2)
    sinal_ctx = fatia_janela(dados, fs, ctx_ini, ctx_fim)[:, canal_idx].astype(float)
    _, _, fm_teta, _, _, fm_gamma = ajusta_fooof_teta_gamma(sinal_ctx, fs)
    fit_range_gamma = (35.0, min(250.0, fs * 0.5 * 0.95))

    # --- comodulograma da janela de 10s (usa seg_limpo, com notch) ----------
    z_mapa, mi_obs, mi_surr = calcula_comodulograma_z(
        seg_limpo, fs, FASES_DEFAULT, AMPS_DEFAULT, n_surr=N_SURR, n_bins=N_BINS, retorna_mi=True)
    p_mapa = p_valores_por_celula(mi_obs, mi_surr)
    sig = bh_fdr_mapa(p_mapa, alpha=FDR_Q)
    z_mapa_plot = np.where(sig, z_mapa, np.nan)
    z_pico, fp_pico, fa_pico = z_pico_par(z_mapa, FASES_DEFAULT, AMPS_DEFAULT, par=par_ativo)
    pico_ok = z_pico is not None and not (isinstance(z_pico, float) and np.isnan(z_pico))

    # --- timeline: envelope de teta + MI em janela deslizante curta ---------
    # usa a fase_pico_hz/amp_pico_hz do PROPRIO evento (mesmo par que gerou
    # z_score_refinado), nao o pico recalculado do comodulograma acima.
    env_teta = np.abs(hilbert(seg_teta))
    t_mi, mi_vals = mi_timeline(seg_limpo, fs, fp_evento, fa_evento)

    # --- figura 5 paineis, grade de 4 colunas: --------------------------
    # linha 0: LFP (2 col) | Timeline teta+MI (2 col) -- mesmo tamanho
    # linha 1: FOOOF teta (2 col) | FOOOF gama (2 col) -- mesmo tamanho da
    #          linha 0 (mesma divisao de colunas)
    # linha 2: Comodulograma (2 col do meio) -- reduzido e centralizado
    fig = plt.figure(figsize=(15, 15))
    gs = fig.add_gridspec(3, 4, height_ratios=[1.0, 1.0, 1.1])
    ax1 = fig.add_subplot(gs[0, 0:2])
    ax5 = fig.add_subplot(gs[0, 2:4])
    ax2 = fig.add_subplot(gs[1, 0:2])
    ax3 = fig.add_subplot(gs[1, 2:4])
    ax4 = fig.add_subplot(gs[2, 1:3])
    fig.suptitle(
        f"{row['rato']} | {row['comportamento']} | canal {int(row['canal'])} | {par_ativo} | "
        f"z_dataset={row['z_score_refinado']:.2f} | {row['arquivo']} "
        f"[{janela_ini_local:.0f}-{janela_fim_local:.0f}s]",
        fontsize=12,
    )

    # painel 1: LFP bruto + teta + gama (empilhados verticalmente)
    off = 3.0 * np.std(seg_limpo)
    ax1.plot(t_seg, seg_limpo, color="#2c3e50", lw=0.8, label="Bruto (notch 60Hz)")
    ax1.plot(t_seg, seg_teta - off, color="#2980b9", lw=1.0, label="Teta 4-8Hz")
    ax1.plot(t_seg, seg_gama - 2 * off, color="#c0392b", lw=0.8, label="Gama 30-80Hz")
    ax1.set_title("LFP -- janela de 10s")
    ax1.set_xlabel("Tempo (s)")
    ax1.set_xlim(0, 10)
    ax1.set_yticks([])
    ax1.legend(fontsize=8, loc="upper right")
    ax1.grid(alpha=0.2)

    # painel 5 (ao lado do LFP): timeline teta (envelope) + MI em janela
    # deslizante curta, mesmo eixo de tempo do painel 1 -- so descritivo
    # (MI cru, sem surrogates), ajuda a localizar ONDE na janela de 10s a
    # modulacao se concentra (nao da pra ler atraso/velocidade em ms com
    # confianca: o envelope e instantaneo, o MI e sempre uma media sobre a
    # janela).
    ax5b = ax5.twinx()
    ax5.plot(t_seg, env_teta, color="#2980b9", lw=1.2)
    ax5.set_ylabel("Envelope teta", color="#2980b9")
    ax5.tick_params(axis="y", labelcolor="#2980b9")
    ax5b.plot(t_mi, mi_vals, color="#c0392b", lw=1.5, marker="o", ms=3)
    ax5b.set_ylabel(f"MI cru ({par_ativo}, janela 0.75s)", color="#c0392b")
    ax5b.tick_params(axis="y", labelcolor="#c0392b")
    ax5.set_xlabel("Tempo (s)")
    ax5.set_xlim(0, 10)
    ax5.set_title("Timeline descritiva -- teta x MI (janela 0.75s/passo 0.1s), sem teste estatistico", fontsize=9)
    ax5.grid(alpha=0.2)

    # paineis 2/3: FOOOF teta e gama/HG (reusa logica ja validada, nao
    # duplicada aqui) -- Y igualado nos dois depois, usando como referencia
    # o maior valor entre os dois (pedido do usuario: comparar escalas).
    painel_fooof(ax2, fm_teta, cor_ap="#2980b9", cor_flat="#27ae60",
                cor_pico="#e74c3c", faixa_pico=(4.0, 12.0), rotulo="Teta",
                fit_range=FOOOF_TETA_FIT_RANGE)
    painel_fooof(ax3, fm_gamma, cor_ap="#8e44ad", cor_flat="#e67e22",
                cor_pico="#d35400", faixa_pico=fit_range_gamma, rotulo="Gama/HG",
                fit_range=fit_range_gamma)
    y2_min, y2_max = ax2.get_ylim()
    y3_min, y3_max = ax3.get_ylim()
    y_min_comum, y_max_comum = min(y2_min, y3_min), max(y2_max, y3_max)
    ax2.set_ylim(y_min_comum, y_max_comum)
    ax3.set_ylim(y_min_comum, y_max_comum)

    # painel 4: comodulograma -- reduzido (2 das 4 colunas) e centralizado
    # (nao encosta na esquerda nem na direita). Branco = celula que NAO
    # passou na correcao FDR (nao e "sem dado", e "nao significativo" --
    # e assim que a "ilha" aparece isolada do fundo, ver checklist).
    # vmin/vmax vem do mapa BRUTO (antes da mascara de FDR), nao do
    # mascarado: se nenhuma celula sobreviver ao FDR nesta execucao (mapa
    # 100% NaN -- pode acontecer, surrogates aqui nao tem seed fixa), o
    # matplotlib nao tem como inferir vmin/vmax sozinho e cai num intervalo
    # arbitrario tipo -0.1/0.1 que nao tem nada a ver com a escala real de z.
    vmin_comod = min(0.0, float(np.nanmin(z_mapa)))
    vmax_comod = max(3.0, float(np.nanmax(z_mapa)))
    im = ax4.pcolormesh(FASES_DEFAULT, AMPS_DEFAULT, z_mapa_plot, shading="auto",
                        cmap="viridis", vmin=vmin_comod, vmax=vmax_comod)
    if not sig.any():
        ax4.text(0.5, 0.5, "nenhuma celula sobreviveu ao FDR nesta execucao",
                 transform=ax4.transAxes, ha="center", va="center",
                 fontsize=10, color="#888888")
    if pico_ok:
        ax4.scatter([fp_pico], [fa_pico], s=80, c="red", marker="X",
                   edgecolors="white", linewidths=1.5)
        titulo_comod = f"Comodulograma 10s (pico {par_ativo} z={z_pico:.2f})"
    else:
        titulo_comod = f"Comodulograma 10s (sem pico {par_ativo} significativo)"
    ax4.set_xlabel("Fase (Hz)")
    ax4.set_ylabel("Amplitude (Hz)")
    ax4.set_title(titulo_comod + "\n(branco = nao-significativo apos FDR, nao ausencia de dado)", fontsize=9)
    fig.colorbar(im, ax=ax4, label="z", fraction=0.04, pad=0.03)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    return fig


In [24]:
# CELULA 4: CONTROLES INTERATIVOS (Rato -> Comportamento -> Janela campeã)

def _comportamentos_do_rato(rato):
    sub = df_eventos[df_eventos["rato"] == rato]
    return sorted(sub["comportamento"].dropna().unique())


def _opcoes_janela(rato, comportamento):
    sub = df_eventos[(df_eventos["rato"] == rato) & (df_eventos["comportamento"] == comportamento)]
    sub = sub.sort_values("z_score_refinado", ascending=False)
    opcoes = []
    for idx, r in sub.iterrows():
        rotulo = (f"z={r['z_score_refinado']:.2f} | canal {int(r['canal'])} | {r['par']} | "
                 f"{r['arquivo']} [{r['janela_ini_s']:.0f}-{r['janela_fim_s']:.0f}s]")
        opcoes.append((rotulo, idx))
    return opcoes


ratos_disponiveis = sorted(df_eventos["rato"].dropna().unique())

rato_dd = widgets.Dropdown(options=ratos_disponiveis, description="Rato:", style={"description_width": "100px"})
comport_dd = widgets.Dropdown(description="Comportamento:", style={"description_width": "100px"})
janela_dd = widgets.Dropdown(description="Janela campeã:", style={"description_width": "100px"},
                             layout=widgets.Layout(width="500px"))
btn = widgets.Button(description="Visualizar", button_style="primary", icon="area-chart")
out = widgets.Output()


def _on_rato_change(change):
    opcoes = _comportamentos_do_rato(rato_dd.value)
    comport_dd.options = opcoes
    if opcoes:
        comport_dd.value = opcoes[0]


def _on_comport_change(change):
    janela_dd.options = _opcoes_janela(rato_dd.value, comport_dd.value)


def _on_click(_botao):
    with out:
        clear_output(wait=True)
        if janela_dd.value is None:
            print("Nenhuma janela disponível para esse filtro.")
            return
        row = df_eventos.loc[janela_dd.value]
        try:
            plota_dissecacao_completa(row)
        except Exception as e:
            print(f"Erro ao gerar visualização: {e}")


rato_dd.observe(_on_rato_change, names="value")
comport_dd.observe(_on_comport_change, names="value")
btn.on_click(_on_click)

_on_rato_change(None)
_on_comport_change(None)

display(widgets.VBox([
    widgets.HBox([rato_dd, comport_dd]),
    janela_dd,
    btn,
    out,
]))
